
# Amazon Search — Week 7 V1.1
## Corrected End-to-End Retrieval + LambdaMART Pipeline

This revision fixes the main V1 integration problem:

- Full retrieval catalog: Week 1 `cleaned_products_10k.parquet`
- SBERT retrieval embeddings: Week 4 V1
- CLIP image embeddings: Week 4 V2
- Runtime ranking features: Week 2 + Week 5 definitions
- Saved LambdaMART: Week 5 V2, with exact saved feature order

### Upload these 5 ZIP files
1. `amazon_week1_v4_processed_data*.zip`
2. `amazon_week4_v1_sbert_faiss_results*.zip`
3. `amazon_week4_v2_clip_multimodal_results*.zip`
4. `amazon_week5_v1_feature_dataset.zip`
5. `amazon_week5_v2_lambdamart_results.zip`

Pipeline:

`Query → SBERT → FAISS Top-100 → Week5-compatible features → LambdaMART → Top-K`


In [ ]:

# 1. Install dependencies
!pip -q install pandas pyarrow numpy scikit-learn faiss-cpu lightgbm sentence-transformers rank-bm25 transformers accelerate

print("✅ Dependencies installed")


In [ ]:

# 2. Upload and extract required ZIPs
from google.colab import files
from pathlib import Path
import zipfile, shutil, os, json, re, warnings, time
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path("/content/amazon_week7_v1_1")
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True, exist_ok=True)

UPLOAD = ROOT / "uploads"
EXTRACT = ROOT / "extracted"
OUT = ROOT / "outputs"
for d in [UPLOAD, EXTRACT, OUT]:
    d.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
for name, payload in uploaded.items():
    (UPLOAD / name).write_bytes(payload)

def pick_zip(tokens):
    for p in UPLOAD.glob("*.zip"):
        low = p.name.lower()
        if all(t.lower() in low for t in tokens):
            return p
    return None

zips = {
    "week1": pick_zip(["week1", "processed"]),
    "week4_v1": pick_zip(["week4", "v1", "sbert"]),
    "week4_v2": pick_zip(["week4", "v2", "clip"]),
    "week5_v1": pick_zip(["week5", "v1", "feature"]),
    "week5_v2": pick_zip(["week5", "v2", "lambda"]),
}

missing = [k for k,v in zips.items() if v is None]
if missing:
    print(zips)
    raise ValueError("Missing ZIP(s): " + ", ".join(missing))

for key, zp in zips.items():
    dest = EXTRACT / key
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zp, "r") as z:
        z.extractall(dest)
    print("✅", key, "->", zp.name)

def must_find(folder, filename):
    hits = list(folder.rglob(filename))
    if not hits:
        hits = [p for p in folder.rglob("*") if p.is_file() and p.name.lower() == filename.lower()]
    if not hits:
        raise FileNotFoundError(f"{filename} not found under {folder}")
    return hits[0]

W1 = EXTRACT / "week1"
W4V1 = EXTRACT / "week4_v1"
W4V2 = EXTRACT / "week4_v2"
W5V1 = EXTRACT / "week5_v1"
W5V2 = EXTRACT / "week5_v2"


In [ ]:

# 3. Load the correct full catalog and prior artifacts
catalog = pd.read_parquet(must_find(W1, "cleaned_products_10k.parquet")).reset_index(drop=True)

for c in ["product_id", "title", "description", "category", "brand"]:
    if c not in catalog.columns:
        catalog[c] = ""
    catalog[c] = catalog[c].fillna("").astype(str)

for c in ["price", "average_rating", "review_count"]:
    if c not in catalog.columns:
        catalog[c] = np.nan
    catalog[c] = pd.to_numeric(catalog[c], errors="coerce")

catalog["product_id"] = catalog["product_id"].astype(str)
catalog["document"] = (catalog["title"] + " " + catalog["description"]).str.strip()

sbert_product_emb = np.load(
    must_find(W4V1, "week4_sbert_product_embeddings.npy")
).astype(np.float32)

if len(sbert_product_emb) != len(catalog):
    raise ValueError(f"SBERT/catalog mismatch: {len(sbert_product_emb)} vs {len(catalog)}")

sbert_product_emb /= np.clip(
    np.linalg.norm(sbert_product_emb, axis=1, keepdims=True),
    1e-12, None
)

clip_img_emb = np.load(
    must_find(W4V2, "week4_clip_image_embeddings.npy")
).astype(np.float32)

clip_map = pd.read_csv(
    must_find(W4V2, "week4_clip_embedding_map.csv")
)
clip_map["product_id"] = clip_map["product_id"].astype(str)
if "clip_row" not in clip_map.columns:
    clip_map["clip_row"] = np.arange(len(clip_map))

if len(clip_img_emb) != len(clip_map):
    raise ValueError(f"CLIP embedding/map mismatch: {len(clip_img_emb)} vs {len(clip_map)}")

clip_img_emb /= np.clip(
    np.linalg.norm(clip_img_emb, axis=1, keepdims=True),
    1e-12, None
)
product_to_clip_row = dict(zip(
    clip_map["product_id"],
    clip_map["clip_row"].astype(int)
))

w5_dataset = pd.read_parquet(
    must_find(W5V1, "week5_feature_dataset.parquet")
)

with open(must_find(W5V1, "week5_feature_list.json"), "r") as f:
    feature_cols_v1 = json.load(f)

with open(must_find(W5V2, "week5_v2_config.json"), "r") as f:
    w5_config = json.load(f)

feature_cols = w5_config.get("feature_cols", feature_cols_v1)
model_path = must_find(W5V2, "week5_v2_lambdamart_model.txt")

print(f"✅ Catalog rows: {len(catalog):,}")
print("✅ SBERT embeddings:", sbert_product_emb.shape)
print("✅ CLIP embeddings:", clip_img_emb.shape)
print(f"✅ LambdaMART feature count: {len(feature_cols)}")
for i,c in enumerate(feature_cols,1):
    print(f"{i:02d}. {c}")


In [ ]:

# 4. Load models and build full-catalog FAISS index
import faiss
import lightgbm as lgb
import torch

from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPTokenizerFast
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

SBERT_NAME = "sentence-transformers/all-MiniLM-L6-v2"
CLIP_NAME = "openai/clip-vit-base-patch32"

sbert_model = SentenceTransformer(SBERT_NAME)

faiss_index = faiss.IndexFlatIP(sbert_product_emb.shape[1])
faiss_index.add(sbert_product_emb.astype(np.float32))

ranker = lgb.Booster(model_file=str(model_path))

if ranker.num_feature() != len(feature_cols):
    raise ValueError(
        f"Model expects {ranker.num_feature()} features, schema has {len(feature_cols)}"
    )

clip_model = None
clip_tokenizer = None
clip_device = None

def ensure_clip():
    global clip_model, clip_tokenizer, clip_device
    if clip_model is None:
        clip_device = "cuda" if torch.cuda.is_available() else "cpu"
        print("Loading CLIP on", clip_device)
        clip_model = CLIPModel.from_pretrained(CLIP_NAME).to(clip_device)
        clip_tokenizer = CLIPTokenizerFast.from_pretrained(CLIP_NAME)
        clip_model.eval()
    return clip_model, clip_tokenizer, clip_device

print("✅ FAISS ntotal:", faiss_index.ntotal)
print("✅ LambdaMART num_feature:", ranker.num_feature())


In [ ]:

# 5. Week2 / Week5-compatible feature builder
def tokenize(text):
    return re.findall(r"[a-z0-9]+", str(text).lower())

def norm_text(x):
    x = "" if pd.isna(x) else str(x).lower()
    x = re.sub(r"[^a-z0-9\s]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x

def overlap_ratio(q, text):
    qset = set(tokenize(q))
    if not qset:
        return 0.0
    return len(qset & set(tokenize(text))) / len(qset)

def token_overlap(a, b):
    A, B = set(norm_text(a).split()), set(norm_text(b).split())
    if not A:
        return 0.0
    return len(A & B) / len(A)

def minmax_array(x):
    x = np.asarray(x, dtype=np.float32)
    if len(x) == 0:
        return x
    lo = np.nanmin(x)
    hi = np.nanmax(x)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros(len(x), dtype=np.float32)
    return ((x - lo) / (hi - lo)).astype(np.float32)

train_w5 = w5_dataset[w5_dataset["split"].astype(str).eq("train")]

train_medians = {}
for c in ["price", "average_rating", "review_count"]:
    if c in train_w5.columns:
        med = pd.to_numeric(train_w5[c], errors="coerce").median()
        train_medians[c] = 0.0 if pd.isna(med) else float(med)
    else:
        train_medians[c] = 0.0

def encode_clip_query(query):
    model, tok, device = ensure_clip()
    with torch.no_grad():
        batch = tok(
            [query],
            padding=True,
            truncation=True,
            max_length=77,
            return_tensors="pt"
        ).to(device)
        feat = model.get_text_features(**batch)
        feat = feat / feat.norm(dim=-1, keepdim=True)
    return feat.cpu().numpy().astype(np.float32)[0]

def build_runtime_features(query, candidate_idx, sbert_scores):
    candidate_idx = np.asarray(candidate_idx, dtype=int)
    cand = catalog.iloc[candidate_idx].copy().reset_index(drop=False)
    docs = cand["document"].tolist()

    # Week2 lexical features: fit within candidate set
    tokenized_docs = [tokenize(x) for x in docs]
    bm25 = BM25Okapi(tokenized_docs)
    bm25_score = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)

    vec = TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=1)
    Xtf = vec.fit_transform([query] + docs)
    tfidf_similarity = cosine_similarity(Xtf[0:1], Xtf[1:]).ravel().astype(np.float32)

    raw_price = pd.to_numeric(cand["price"], errors="coerce")
    raw_rating = pd.to_numeric(cand["average_rating"], errors="coerce")
    raw_reviews = pd.to_numeric(cand["review_count"], errors="coerce")

    price = raw_price.fillna(train_medians["price"]).to_numpy(np.float32)
    rating = raw_rating.fillna(train_medians["average_rating"]).to_numpy(np.float32)
    reviews = raw_reviews.fillna(train_medians["review_count"]).to_numpy(np.float32)
    reviews = np.clip(reviews, 0, None)

    query_len = np.full(len(cand), len(tokenize(query)), dtype=np.float32)
    title_len = np.array([len(tokenize(x)) for x in cand["title"]], dtype=np.float32)
    description_len = np.array([len(tokenize(x)) for x in cand["description"]], dtype=np.float32)

    query_title_overlap = np.array(
        [overlap_ratio(query, x) for x in cand["title"]], dtype=np.float32
    )
    query_description_overlap = np.array(
        [overlap_ratio(query, x) for x in cand["description"]], dtype=np.float32
    )

    sbert_similarity = np.asarray(sbert_scores, dtype=np.float32)
    sbert_available = np.ones(len(cand), dtype=np.float32)

    q_clip = encode_clip_query(query)
    clip_similarity = np.zeros(len(cand), dtype=np.float32)
    clip_available = np.zeros(len(cand), dtype=np.float32)

    for i, pid in enumerate(cand["product_id"].astype(str)):
        row = product_to_clip_row.get(pid)
        if row is not None and 0 <= int(row) < len(clip_img_emb):
            clip_similarity[i] = float(np.dot(q_clip, clip_img_emb[int(row)]))
            clip_available[i] = 1.0

    q_norm = norm_text(query)
    title_norm = cand["title"].map(norm_text)
    brand_norm = cand["brand"].map(norm_text)

    exact_title_match = (title_norm == q_norm).astype(np.float32).to_numpy()
    query_in_title = np.array(
        [float(bool(q_norm) and q_norm in t) for t in title_norm], dtype=np.float32
    )
    query_title_token_coverage = np.array(
        [token_overlap(query, t) for t in cand["title"]], dtype=np.float32
    )
    brand_in_query = np.array(
        [float(bool(b) and b in q_norm) for b in brand_norm], dtype=np.float32
    )
    category_token_match = np.array(
        [token_overlap(query, c) for c in cand["category"]], dtype=np.float32
    )

    log_review_count = np.log1p(reviews).astype(np.float32)
    log_price = np.log1p(np.clip(price, 0, None)).astype(np.float32)

    data = {
        "bm25_score": bm25_score,
        "tfidf_similarity": tfidf_similarity,
        "price": price,
        "average_rating": rating,
        "log_review_count": log_review_count,
        "query_len": query_len,
        "title_len": title_len,
        "description_len": description_len,
        "query_title_overlap": query_title_overlap,
        "query_description_overlap": query_description_overlap,
        "sbert_similarity": sbert_similarity,
        "clip_similarity": clip_similarity,
        "sbert_available": sbert_available,
        "clip_available": clip_available,
        "log_price": log_price,
        "log_review_count_w5": log_review_count,
        "rating_missing_w5": np.zeros(len(cand), dtype=np.float32),
        "price_missing_w5": np.zeros(len(cand), dtype=np.float32),
        "exact_title_match": exact_title_match,
        "query_in_title": query_in_title,
        "query_title_token_coverage": query_title_token_coverage,
        "brand_in_query": brand_in_query,
        "category_token_match": category_token_match,
        "bm25_score_qnorm": minmax_array(bm25_score),
        "tfidf_similarity_qnorm": minmax_array(tfidf_similarity),
        "sbert_similarity_qnorm": minmax_array(sbert_similarity),
        "clip_similarity_qnorm": minmax_array(clip_similarity),
    }

    X = pd.DataFrame(data)

    missing_runtime = [c for c in feature_cols if c not in X.columns]
    if missing_runtime:
        raise ValueError("Unimplemented model features: " + ", ".join(missing_runtime))

    X = X[feature_cols].copy()
    for c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0.0)

    return cand, X


In [ ]:

# 6. Correct end-to-end search
from dataclasses import dataclass, asdict

@dataclass
class SearchLatency:
    sbert_embedding_ms: float
    faiss_retrieval_ms: float
    feature_generation_ms: float
    lambdamart_ms: float
    total_ms: float

def search_products(query, top_k=10, candidate_k=100):
    t_total = time.perf_counter()

    t0 = time.perf_counter()
    q_emb = sbert_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype(np.float32)
    sbert_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    k = min(int(candidate_k), len(catalog))
    scores, idx = faiss_index.search(q_emb, k)
    candidate_idx = idx[0]
    sbert_scores = scores[0]
    valid = candidate_idx >= 0
    candidate_idx = candidate_idx[valid]
    sbert_scores = sbert_scores[valid]
    retrieval_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    cand, X = build_runtime_features(query, candidate_idx, sbert_scores)
    feature_ms = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    pred = np.asarray(ranker.predict(X), dtype=np.float32)
    ltr_ms = (time.perf_counter() - t0) * 1000

    order = np.argsort(-pred)[:top_k]

    show_cols = [
        "product_id","title","description","category","brand",
        "price","average_rating","review_count"
    ]
    show_cols = [c for c in show_cols if c in cand.columns]

    result = cand.iloc[order][show_cols].copy().reset_index(drop=True)
    result.insert(0, "rank", np.arange(1, len(result)+1))
    result["lambdamart_score"] = pred[order]
    result["sbert_retrieval_score"] = sbert_scores[order]

    total_ms = (time.perf_counter() - t_total) * 1000

    latency = SearchLatency(
        sbert_embedding_ms=sbert_ms,
        faiss_retrieval_ms=retrieval_ms,
        feature_generation_ms=feature_ms,
        lambdamart_ms=ltr_ms,
        total_ms=total_ms
    )
    return result, latency

print("✅ search_products() ready")



## Quality smoke test

The first test uses queries already present in the Week 5 judged dataset.  
This avoids judging the system only on arbitrary queries that may not exist in the 10K sample catalog.


In [ ]:

# 7. Known-query smoke test
query_table = (
    w5_dataset[["query_id","query"]]
    .drop_duplicates("query_id")
    .dropna()
    .reset_index(drop=True)
)
query_table["n_tokens"] = query_table["query"].astype(str).map(lambda x: len(tokenize(x)))

known_queries = (
    query_table[query_table["n_tokens"].between(2,6)]
    .head(3)["query"]
    .astype(str)
    .tolist()
)
if len(known_queries) < 3:
    known_queries = query_table.head(3)["query"].astype(str).tolist()

print("Known queries:", known_queries)

sample_rows = []
lat_rows = []

for q in known_queries:
    print("\n" + "="*100)
    print("QUERY:", q)
    res, lat = search_products(q, top_k=10, candidate_k=100)
    display(res)
    temp = res.copy()
    temp.insert(0, "query", q)
    sample_rows.append(temp)
    lat_rows.append({"query": q, **asdict(lat)})

known_results = pd.concat(sample_rows, ignore_index=True)
known_latency = pd.DataFrame(lat_rows)
display(known_latency)


In [ ]:

# 8. Optional arbitrary-query demo
DEMO_QUERY = "wireless gaming mouse"

demo_results, demo_latency = search_products(DEMO_QUERY, top_k=10, candidate_k=100)
print("QUERY:", DEMO_QUERY)
display(demo_results)
print(asdict(demo_latency))


In [ ]:

# 9. Warm latency benchmark
bench_queries = known_queries.copy()
for q in query_table.iloc[3:15]["query"].astype(str):
    if q not in bench_queries:
        bench_queries.append(q)
bench_queries = bench_queries[:10]

# Warmup includes CLIP model initialization.
_ = search_products(bench_queries[0], top_k=10, candidate_k=100)

bench_rows = []
for q in bench_queries:
    _, lat = search_products(q, top_k=10, candidate_k=100)
    bench_rows.append({"query": q, **asdict(lat)})

bench = pd.DataFrame(bench_rows)
display(bench)

latency_summary = bench.drop(columns=["query"]).agg(["mean","median","min","max"]).T
latency_summary.columns = ["mean_ms","median_ms","min_ms","max_ms"]
display(latency_summary)

mean_total = float(bench["total_ms"].mean())
p95_total = float(bench["total_ms"].quantile(0.95))

print(f"Mean total: {mean_total:.2f} ms")
print(f"P95 total : {p95_total:.2f} ms")
print("Target <200ms:", "✅ PASS" if mean_total < 200 else "⚠️ NOT YET")


In [ ]:

# 10. Integration checks
def validate_integration():
    checks = {
        "catalog_is_10k_scale": len(catalog) >= 9000,
        "sbert_catalog_alignment": len(sbert_product_emb) == len(catalog),
        "faiss_catalog_alignment": faiss_index.ntotal == len(catalog),
        "model_feature_count_match": ranker.num_feature() == len(feature_cols),
    }

    q = known_queries[0]
    q_emb = sbert_model.encode(
        [q], convert_to_numpy=True, normalize_embeddings=True
    ).astype(np.float32)
    scores, idx = faiss_index.search(q_emb, min(100, len(catalog)))
    cand, X = build_runtime_features(q, idx[0], scores[0])

    checks["runtime_feature_order_match"] = list(X.columns) == list(feature_cols)
    checks["runtime_feature_count_match"] = X.shape[1] == ranker.num_feature()
    checks["runtime_features_finite"] = bool(np.isfinite(X.to_numpy()).all())

    pred = ranker.predict(X)
    checks["saved_lambdamart_predicts"] = bool(
        len(pred) == len(X) and np.isfinite(pred).all()
    )

    for k,v in checks.items():
        print(("✅" if v else "❌"), k)

    if not all(checks.values()):
        raise AssertionError("Failed: " + ", ".join(k for k,v in checks.items() if not v))

    print("\n✅ WEEK 7 V1.1 CORE INTEGRATION PASSED")
    return checks

checks = validate_integration()


In [ ]:

# 11. Save outputs
from datetime import datetime

known_results.to_csv(OUT / "week7_v1_1_known_query_ranked_results.csv", index=False)
known_latency.to_csv(OUT / "week7_v1_1_known_query_latency.csv", index=False)
demo_results.to_csv(OUT / "week7_v1_1_demo_ranked_results.csv", index=False)
bench.to_csv(OUT / "week7_v1_1_latency_benchmark.csv", index=False)
latency_summary.to_csv(OUT / "week7_v1_1_latency_summary.csv")

pd.DataFrame({
    "feature_order": np.arange(1, len(feature_cols)+1),
    "feature": feature_cols
}).to_csv(OUT / "week7_v1_1_runtime_feature_schema.csv", index=False)

summary = {
    "week": 7,
    "version": "V1.1",
    "catalog_rows": int(len(catalog)),
    "catalog_source": "Week1 cleaned_products_10k.parquet",
    "retrieval_source": "Week4 V1 SBERT embeddings + FAISS",
    "clip_source": "Week4 V2 image embeddings",
    "ranker_source": "Week5 V2 saved LambdaMART model",
    "ranker_features": int(len(feature_cols)),
    "candidate_k": 100,
    "top_k": 10,
    "mean_total_latency_ms": mean_total,
    "p95_total_latency_ms": p95_total,
    "latency_target_ms": 200,
    "latency_target_pass": bool(mean_total < 200),
    "integration_checks": {k: bool(v) for k,v in checks.items()},
    "generated_at_utc": datetime.utcnow().isoformat() + "Z"
}

with open(OUT / "week7_v1_1_integration_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


In [ ]:

# 12. Package outputs
zip_base = "/content/amazon_week7_v1_1_corrected_results"
zip_path = shutil.make_archive(zip_base, "zip", OUT)
print("✅ Created:", zip_path)
files.download(zip_path)



## V1.1 acceptance criteria

Move to Week 7 V2 only if:

- catalog is approximately 10K products;
- FAISS/SBERT is aligned to that catalog;
- runtime feature order exactly matches Week 5;
- saved LambdaMART is actually used;
- sample ranking quality is semantically reasonable;
- latency is measured honestly.

Then V2 can wrap the correct pipeline in a FastAPI `/search` endpoint.
